## ML_1024: KV Cache Attention

**Difficulty**: Hard | **TorchCode**: #14

### Problem
Implement multi-head attention with a KV cache for efficient autoregressive decoding. During prefill, apply a causal mask. During decode, append new K/V to the cache and attend over the full history.

### Formula
$$K_{\text{total}} = [K_{\text{cache}};\, K_{\text{new}}], \quad \text{out}, \text{cache} = \text{MHA}(Q_{\text{new}}, K_{\text{total}}, V_{\text{total}})$$

In [ ]:
import torch
import torch.nn as nn
import math

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, cache=None):
        B, S_new, _ = x.shape
        q = self.W_q(x).view(B, S_new, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, S_new, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, S_new, self.num_heads, self.d_k).transpose(1, 2)
        if cache is not None:
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        new_cache = (k, v)
        S_total = k.shape[2]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if S_new > 1:
            S_past = S_total - S_new
            mask = torch.triu(
                torch.ones(S_new, S_total, device=x.device, dtype=torch.bool),
                diagonal=S_past + 1,
            )
            scores = scores.masked_fill(mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        attn = torch.matmul(weights, v)
        out = self.W_o(attn.transpose(1, 2).contiguous().view(B, S_new, -1))
        return out, new_cache

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Output shape (no cache) ---
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
assert isinstance(attn, nn.Module)
out, cache = attn(torch.randn(2, 8, 64))
assert out.shape == (2, 8, 64)

# --- Test 2: Cache structure ---
torch.manual_seed(0)
attn2 = KVCacheAttention(d_model=64, num_heads=4)
out, cache = attn2(torch.randn(2, 8, 64))
assert isinstance(cache, tuple) and len(cache) == 2
assert cache[0].shape == (2, 4, 8, 16)
assert cache[1].shape == (2, 4, 8, 16)

# --- Test 3: Decode step appends to cache ---
torch.manual_seed(0)
attn3 = KVCacheAttention(d_model=32, num_heads=2)
_, cache = attn3(torch.randn(1, 4, 32))
out, new_cache = attn3(torch.randn(1, 1, 32), cache=cache)
assert out.shape == (1, 1, 32)
assert new_cache[0].shape[2] == 5

# --- Test 4: Incremental decode matches full forward ---
torch.manual_seed(42)
attn4 = KVCacheAttention(d_model=32, num_heads=2)
x = torch.randn(1, 6, 32)
full_out, _ = attn4(x)
out1, cache = attn4(x[:, :4])
out2, cache = attn4(x[:, 4:5], cache=cache)
out3, cache = attn4(x[:, 5:6], cache=cache)
assert torch.allclose(full_out, torch.cat([out1, out2, out3], dim=1), atol=1e-5)

# --- Test 5: Gradient flow ---
torch.manual_seed(0)
attn5 = KVCacheAttention(d_model=32, num_heads=2)
x = torch.randn(1, 4, 32, requires_grad=True)
attn5(x)[0].sum().backward()
assert x.grad is not None

print('All tests passed!')

## Method2

In [1]:
import torch
import torch.nn as nn
import math


class KVCacheCausalGroupQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int, max_seq_len: int):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.repeats = num_heads // num_kv_heads
        self.max_seq_len = max_seq_len

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

        # cache will be created/reset later
        self.cache_k = None   # shape: (B, num_kv_heads, max_seq_len, d_k)
        self.cache_v = None   # shape: (B, num_kv_heads, max_seq_len, d_k)
        self.cache_len = 0

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.cache_len = 0

    def repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, num_kv_heads, S, d_k)
        return: (B, num_heads, S, d_k)
        """
        B, H_kv, S, D = x.shape
        x = x.unsqueeze(2)                                   # (B, H_kv, 1, S, D)
        x = x.expand(B, H_kv, self.repeats, S, D)           # (B, H_kv, repeats, S, D)
        x = x.contiguous().view(B, H_kv * self.repeats, S, D)
        return x

    def softmax_manual(self, x: torch.Tensor, dim: int = -1) -> torch.Tensor:
        max_value = x.max(dim=dim, keepdim=True).values
        x = x - max_value
        exp_x = torch.exp(x)
        sum_exp = exp_x.sum(dim=dim, keepdim=True)
        return exp_x / sum_exp

    def build_causal_mask(self, S_q: int, S_k: int, device, q_start_pos: int = 0) -> torch.Tensor:
        """
        Build causal mask for prefill or general case.

        q positions are:
            q_start_pos, q_start_pos+1, ..., q_start_pos+S_q-1
        k positions are:
            0, 1, ..., S_k-1

        allow if key_pos <= query_pos
        """
        q_pos = torch.arange(q_start_pos, q_start_pos + S_q, device=device).unsqueeze(1)  # (S_q, 1)
        k_pos = torch.arange(S_k, device=device).unsqueeze(0)                              # (1, S_k)
        mask = k_pos <= q_pos
        return mask  # (S_q, S_k)

    def forward_prefill(self, x: torch.Tensor) -> torch.Tensor:
        """
        Prefill stage: process the whole prompt at once and fill KV cache.

        x: (B, S, d_model)
        return: (B, S, d_model)
        """
        B, S, _ = x.shape
        assert S <= self.max_seq_len

        # projections
        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)       # (B, H,   S, d_k)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)    # (B, Hkv, S, d_k)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)    # (B, Hkv, S, d_k)

        # save to cache
        self.cache_k = torch.zeros(
            B, self.num_kv_heads, self.max_seq_len, self.d_k,
            device=x.device, dtype=x.dtype
        )
        self.cache_v = torch.zeros(
            B, self.num_kv_heads, self.max_seq_len, self.d_k,
            device=x.device, dtype=x.dtype
        )
        self.cache_k[:, :, :S, :] = k
        self.cache_v[:, :, :S, :] = v
        self.cache_len = S

        # attention uses current prompt only
        k_full = self.repeat_kv(k)   # (B, H, S, d_k)
        v_full = self.repeat_kv(v)   # (B, H, S, d_k)

        scores = (q @ k_full.transpose(-2, -1)) / math.sqrt(self.d_k)   # (B, H, S, S)

        mask = self.build_causal_mask(S_q=S, S_k=S, device=x.device, q_start_pos=0)
        scores = scores.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        attn = self.softmax_manual(scores, dim=-1)
        out = attn @ v_full                                                # (B, H, S, d_k)

        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        out = self.W_o(out)
        return out

    def forward_decode(self, x_t: torch.Tensor) -> torch.Tensor:
        """
        Decode one step using KV cache.

        x_t: (B, 1, d_model)  -- current newly generated token embedding
        return: (B, 1, d_model)
        """
        B, S_new, _ = x_t.shape
        assert S_new == 1, "forward_decode expects one token at a time"
        assert self.cache_len < self.max_seq_len

        # current q, k, v
        q = self.W_q(x_t).view(B, 1, self.num_heads, self.d_k).transpose(1, 2)        # (B, H,   1, d_k)
        k_new = self.W_k(x_t).view(B, 1, self.num_kv_heads, self.d_k).transpose(1, 2) # (B, Hkv, 1, d_k)
        v_new = self.W_v(x_t).view(B, 1, self.num_kv_heads, self.d_k).transpose(1, 2) # (B, Hkv, 1, d_k)

        # initialize cache if empty
        if self.cache_k is None or self.cache_v is None:
            self.cache_k = torch.zeros(
                B, self.num_kv_heads, self.max_seq_len, self.d_k,
                device=x_t.device, dtype=x_t.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_kv_heads, self.max_seq_len, self.d_k,
                device=x_t.device, dtype=x_t.dtype
            )
            self.cache_len = 0

        # append new kv into cache
        pos = self.cache_len
        self.cache_k[:, :, pos:pos+1, :] = k_new
        self.cache_v[:, :, pos:pos+1, :] = v_new
        self.cache_len += 1

        # read valid cached kv only
        k_cached = self.cache_k[:, :, :self.cache_len, :]   # (B, Hkv, T, d_k)
        v_cached = self.cache_v[:, :, :self.cache_len, :]   # (B, Hkv, T, d_k)

        # expand kv heads
        k_full = self.repeat_kv(k_cached)   # (B, H, T, d_k)
        v_full = self.repeat_kv(v_cached)   # (B, H, T, d_k)

        # current token attends to all cached previous tokens + itself
        scores = (q @ k_full.transpose(-2, -1)) / math.sqrt(self.d_k)   # (B, H, 1, T)

        # for single-step decode, causal mask is not needed because cache only contains past/current tokens
        attn = self.softmax_manual(scores, dim=-1)
        out = attn @ v_full                                              # (B, H, 1, d_k)

        out = out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        out = self.W_o(out)
        return out

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        """
        Convenience wrapper:
        - use_cache=False -> prefill full sequence
        - use_cache=True  -> decode one token
        """
        if use_cache:
            return self.forward_decode(x)
        return self.forward_prefill(x)


/home/kenneth/CODE/AI_PROJECTS/Leetcode/.venv/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


### prefill with prompt

In [2]:
torch.manual_seed(0)

model = KVCacheCausalGroupQueryAttention(
    d_model=32,
    num_heads=4,
    num_kv_heads=2,
    max_seq_len=16
)

prompt = torch.randn(2, 5, 32)
out_prefill = model.forward_prefill(prompt)
print(out_prefill.shape)   # (2, 5, 32)
print(model.cache_len)     # 5


torch.Size([2, 5, 32])
5


### Decode one token at a time 

In [3]:
new_token = torch.randn(2, 1, 32)
out_step1 = model.forward_decode(new_token)
print(out_step1.shape)     # (2, 1, 32)
print(model.cache_len)     # 6

new_token2 = torch.randn(2, 1, 32)
out_step2 = model.forward_decode(new_token2)
print(out_step2.shape)     # (2, 1, 32)
print(model.cache_len)     # 7


torch.Size([2, 1, 32])
6
torch.Size([2, 1, 32])
7
